# DICOM Metadata Explorer (20 min exercise)

**Goal:** Load DICOM files from a folder, extract key metadata, store in a Pandas DataFrame, and print summary statistics.

**Dataset:** After downloading [SIIM COVID-19](https://www.kaggle.com/competitions/siim-covid19-detection/data), set `DICOM_FOLDER` to your DICOM directory. Demo files are in `data/dicom/`.

In [ ]:
!pip install pydicom pandas -q

In [ ]:
from pathlib import Path
import pandas as pd
import pydicom
from pydicom.errors import InvalidDicomError

# Change this to your SIIM COVID-19 DICOM folder after Kaggle download
DICOM_FOLDER = Path('/Users/likhitayerra/Downloads/data/BrainTumorMRI')

In [ ]:
def extract_metadata(dicom_path: Path) -> dict:
    ds = pydicom.dcmread(dicom_path, stop_before_pixels=True, force=True)
    acquisition_date = getattr(ds, 'AcquisitionDate', None) or getattr(ds, 'StudyDate', None)
    return {
        'file_name': dicom_path.name,
        'modality': getattr(ds, 'Modality', None),
        'patient_age': getattr(ds, 'PatientAge', None),
        'acquisition_date': acquisition_date,
        'body_part_examined': getattr(ds, 'BodyPartExamined', None),
    }


rows = []
for path in sorted(DICOM_FOLDER.rglob('*')):
    if not path.is_file():
        continue
    try:
        rows.append(extract_metadata(path))
    except (InvalidDicomError, PermissionError, OSError):
        pass

df = pd.DataFrame(rows)
df

In [ ]:
print('Total files:', len(df))
print('\nModality counts:')
print(df['modality'].value_counts(dropna=False))
print('\nBody part examined counts:')
print(df['body_part_examined'].value_counts(dropna=False))
print('\nFiles with patient age:', df['patient_age'].notna().sum())
print('Files with acquisition date:', df['acquisition_date'].notna().sum())
print('Unique acquisition dates:', df['acquisition_date'].nunique(dropna=True))

In [ ]:
df.to_csv('/Users/likhitayerra/health/data/dicom_metadata.csv', index=False)
print('Saved to data/dicom_metadata.csv')